In [ ]:
# -*- coding: utf-8 -*-
"""
Measure_roof_edges_agentic_vision_gemini_3.5_flash (Cleaned)
Pipeline:
1) Fetch Image URI from BigQuery
2) Lens Distortion Correction (Gemini 3.5 Flash + Code Exec)
3) Reviewer Model Comparison (Original vs Corrected)
4) Roof Edge Detection & Drawing (Gemini 3.5 Flash + Code Exec) - NO MEASUREMENTS
"""

# ==============================================================================
# Setup & Authenticate
# ==============================================================================
!pip install -q google-genai pandas-gbq matplotlib

from google import genai
from google.genai import types
import pandas_gbq
from google.cloud import storage
import os
import io
from PIL import Image
import matplotlib.pyplot as plt
import textwrap

print("Environment setup complete. (Colab Enterprise uses built-in ADC credentials)")

project_id = "YOUR_PROJECT_ID"

# Auto-detect GCP project if placeholder is unchanged
if project_id == "YOUR_PROJECT_ID":
    try:
        import google.auth
        _, auth_project = google.auth.default()
        if auth_project:
            project_id = auth_project
            print(f"[INFO] Auto-detected Google Cloud Project: {project_id}")
    except Exception:
        print("Unable to auto-detect GCP project")
        pass
client = genai.Client(vertexai=True, project=project_id, location='global')


# ==============================================================================
# Step 1: Get Observation ID and Fetch Image URI
# ==============================================================================
# observation_id = 'o1:8pyvx-2PAvgnsnSzN_q7Xg_2:5001ee' #@param {type:"string"}
observation_id = 'o1:vkmqpCyhedkriBfyh0lAAw_4:5001ee'
dataset_id = 'imagery_insights___us'
urls_table_name = 'pano_observations_latest'
uri_field_name = 'gcs_uri'

print(f"Fetching URI for observation ID: {observation_id}...")
sql_query = f"""
SELECT {uri_field_name}
FROM `{project_id}.{dataset_id}.{urls_table_name}`
WHERE observation_id = '{observation_id}'
"""

df = pandas_gbq.read_gbq(sql_query, project_id=project_id, dialect="standard")

if df.empty:
    raise ValueError("No data found for the specified observation ID.")

image_uri = df.iloc[0][uri_field_name]

# Convert mTLS URL to standard gs:// URI if present
if "storage.mtls.cloud.google.com" in image_uri:
    image_uri = image_uri.replace("https://storage.mtls.cloud.google.com/", "gs://")

print(f"Target Image URI: {image_uri}")

# Helper to download original image for side-by-side comparison later
def download_gcs_blob(gcs_uri, destination_file_name):
    storage_client = storage.Client()
    path_parts = gcs_uri[5:].split('/', 1)
    bucket = storage_client.bucket(path_parts[0])
    blob = bucket.blob(path_parts[1])
    blob.download_to_filename(destination_file_name)

download_gcs_blob(image_uri, "original_image.jpg")
print("Downloaded original_image.jpg locally.")


# ==============================================================================
# Step 2: Lens Distortion Correction (Code Execution)
# ==============================================================================
print("\nStarting Lens Distortion Correction...")

DISTORTION_PROMPT = """Analyze the provided image for lens distortion, specifically barrel or fisheye distortion that might affect the straightness of roof lines.

Instructions:
1. Use code execution to check for lens distortion.
2. If distortion is detected, estimate camera parameters (camera matrix and distortion coefficients) and apply an undistortion algorithm using cv2.getOptimalNewCameraMatrix and cv2.undistort to rectify the image.
3. Ensure the output provides a clear 'undistorted' version of the image.
4. Save the final corrected image as 'corrected_image.jpg'.
"""

def run_lens_correction():
    print(f"  -> Pointing directly to GCS URI: {image_uri}")
    image_part = types.Part.from_uri(file_uri=image_uri, mime_type="image/jpeg")

    print("  -> Sending request to Gemini 3.5 Flash API (Fail-fast enabled, no silent retries)...")
    result = client.models.generate_content(
        model="gemini-3.5-flash",
        contents=[image_part, DISTORTION_PROMPT],
        config=types.GenerateContentConfig(
            temperature=0.0,
            tools=[{"code_execution": {}}]
        ),
    )
    # (NOT TO SUBMIT) occasionally the API doesn't return a response
    print("  -> Received response from API!")
    return result

response = run_lens_correction()

# FIXED: Added safety check for empty candidates or parts
if response.candidates and response.candidates[0].content and response.candidates[0].content.parts:
    # (NOT TO SUBMIT) This is the part where we need to make improvements. The 'corrected_image' was never generated.
    saved = False
    for part in response.candidates[0].content.parts:
        if part.inline_data and part.inline_data.mime_type.startswith('image/'):
            with open("corrected_image.jpg", "wb") as f:
                f.write(part.inline_data.data)
            print("Successfully saved corrected_image.jpg")
            saved = True
    if not saved:
        print("[WARNING] No image part found in response. Check model output below.")
        print(response.text)
else:
    print("[ERROR] Response candidate is empty or blocked.")
    if response.candidates and response.candidates[0].finish_reason:
        print(f"Finish Reason: {response.candidates[0].finish_reason}")

# ==============================================================================
# Step 3: Reviewer Model Comparison (CURRENTLY DISABLED)
# ==============================================================================
"""
print("\nRunning Reviewer Model Comparison...")
# ... [Reviewer model code commented out as requested] ...
"""

# Wait to avoid rate limits
import time
print("Sleeping 15 seconds to avoid rate limits...")
time.sleep(15)

# ==============================================================================
# Step 4: Roof Edge Detection & Drawing (Both Images)
# ==============================================================================
print("\nStarting Roof Edge Detection on both images...")

images_to_process = ["original_image.jpg", "corrected_image.jpg"]

EDGE_PROMPT = '''You are an expert computer vision assistant. Your task is to accurately draw the roof edges of ONLY the *primary house* in the foreground.

**CRITICAL FAILURE AVOIDANCE:** Previously, you drew "shortcuts"—long lines cutting straight across the brick walls, grass, or air just to connect the outer corners of the house. You are NOT drawing a generic polygon around the house. You must trace the actual architectural 3D lines.

Follow this strict operational pipeline using code execution:
1. **Bounding Box Isolation:** Visually locate the primary house roof in the foreground. Define a bounding box `(xmin, ymin, xmax, ymax)` to restrict your search area, ensuring no points go into the sky or background.
2. **Trace VISIBLE Physical Edges Only:** Carefully trace the distinct architectural lines of the roof.
    - **Eaves:** Bottom horizontal edges. (DO NOT connect a far-left eave to a far-right eave if it means the line cuts across a wall).
    - **Ridges:** Top horizontal peaks.
    - **Rakes / Hips:** Slanted outer edges.
    - **Valleys:** Inward-facing intersections of roof planes.
3. **Precise Coordinate Mapping & Sanity Check:** Find the exact (x, y) start and end points for each specific line segment.
    - **SANITY CHECK 1 (No Air-Lines):** For every line segment, ask yourself: "Does this line perfectly follow a visible boundary where roofing material meets air/wall, or does it cut across the middle of a wall or grass?" If it cuts across walls or grass, DELETE IT.
    - **SANITY CHECK 2 (Bounds):** Verify all points are within your house bounding box and NOT in the sky.
4. **Drawing:**
    - Draw boldly colored lines over the edges using PIL or cv2.
    - Draw distinct circles/pointers at the vertices.
    - Label each line (e.g., "Eave", "Ridge", "Hip").
    - Output the final annotated image. DO NOT print measurements.
'''

# (NOT TO SUBMIT) It processed edge detection only for the original image.
# The output generated multiple images with lines and boxes for the original image.
# The output was so large and took significant memory (~77MB). I was able to save
# the notebook with those images but could never re-open them since importing a
# notebook has 20MB size limit
for target_image in images_to_process:
    print(f"\n========================================================")
    print(f"  -> Processing edge detection for: {target_image}")
    print(f"========================================================")

    # Check if image exists (useful if lens correction failed)
    if not os.path.exists(target_image):
        print(f"  -> Warning: {target_image} not found. Skipping.")
        continue

    with open(target_image, "rb") as f:
        target_part = types.Part.from_bytes(data=f.read(), mime_type="image/jpeg")

    print(f"  -> Sending request to Gemini API for {target_image}...")
    edge_response = client.models.generate_content(
        model="gemini-3.5-flash",
        contents=[target_part, EDGE_PROMPT],
        config=types.GenerateContentConfig(
            temperature=0.0,
            tools=[{"code_execution": {}}]
        ),
    )
    print("  -> Received edge detection response!")

    # Output processing & Image display
    for part in edge_response.candidates[0].content.parts:
        if part.text:
            print(part.text)
        if part.code_execution_result:
            print("\n# Code Execution Output:")
            print(part.code_execution_result.output)
        if part.inline_data and part.inline_data.mime_type.startswith('image/'):
            print(f"\nDisplaying Final Annotated Image for {target_image}:")
            display(Image.open(io.BytesIO(part.inline_data.data)))